# Olist Customer Intelligence System — Feature Engineering

This notebook builds all engineered features needed for Stage 2 of the project:  
customer segmentation, predictive machine learning, and model explainability.

**Inputs:** nine cleaned CSV files from `Cleaned_Data/`  
**Outputs:** three feature tables saved to `Feature_Data/`

| Output file | Grain | Purpose |
|---|---|---|
| `order_features.csv` | One row per order | Order-level model input |
| `customer_features.csv` | One row per unique customer | RFM + behavioural features for segmentation / churn |
| `seller_features.csv` | One row per seller | Seller performance features |

Run the cells from top to bottom.

## 1. Imports and output folder

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("Feature_Data", exist_ok=True)
print("Imports done. Feature_Data folder is ready.")

Imports done. Feature_Data folder is ready.


## 2. Load the cleaned datasets

In [2]:
customers   = pd.read_csv("Cleaned_Data/olist_customers_dataset.csv")

orders      = pd.read_csv("Cleaned_Data/olist_orders_dataset.csv",
                          parse_dates=[
                              'order_purchase_timestamp',
                              'order_approved_at',
                              'order_delivered_carrier_date',
                              'order_delivered_customer_date',
                              'order_estimated_delivery_date'
                          ])

order_items = pd.read_csv("Cleaned_Data/olist_order_items_dataset.csv",
                          parse_dates=['shipping_limit_date'])

payments    = pd.read_csv("Cleaned_Data/olist_order_payments_dataset.csv")

reviews     = pd.read_csv("Cleaned_Data/olist_order_reviews_dataset.csv",
                          parse_dates=['review_creation_date', 'review_answer_timestamp'])

products    = pd.read_csv("Cleaned_Data/olist_products_dataset.csv")
sellers     = pd.read_csv("Cleaned_Data/olist_sellers_dataset.csv")
cat_trans   = pd.read_csv("Cleaned_Data/product_category_name_translation.csv")

print("All cleaned datasets loaded.")

All cleaned datasets loaded.


---
## 3. Build a rich order-level base table

The first step is to assemble one wide table with every order and all the facts we know about it.  
Later sections will aggregate this table to the customer and seller grain.

### 3a. Add the customer's unique ID to the orders table

The orders table links to `customer_id`, but that is a per-order identifier.  
We need `customer_unique_id` (the real person) to aggregate orders per customer later.

In [3]:
# Bring customer_unique_id and location into the orders table.
orders = orders.merge(
    customers[['customer_id', 'customer_unique_id',
               'customer_city', 'customer_state', 'customer_zip_code_prefix']],
    on='customer_id',
    how='left'
)

print("orders shape after merge:", orders.shape)
print(orders[['order_id', 'customer_id', 'customer_unique_id', 'customer_state']].head(3))

orders shape after merge: (99441, 12)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

                 customer_unique_id customer_state  
0  7c396fd4830fd04220f754e42b4e5bff             SP  
1  af07308b275d755c9edb36a90c618231             BA  
2  3a653a41f6f9fc3d2a113cf8398680e8             GO  


### 3b. Aggregate order items to the order level

One order can have several items. We summarise them into per-order numbers:
- total product cost, total freight, number of items, number of unique products and sellers.

In [4]:
# Add the English category name to products before aggregating.
products_with_en = products.merge(cat_trans, on='product_category_name', how='left')

# Attach product info to each order item row.
items_rich = order_items.merge(
    products_with_en[['product_id', 'product_category_name',
                       'product_category_name_english', 'product_weight_g']],
    on='product_id',
    how='left'
)

# Aggregate to one row per order.
items_agg = items_rich.groupby('order_id').agg(
    total_items        = ('order_item_id',  'count'),
    unique_products    = ('product_id',     'nunique'),
    unique_sellers     = ('seller_id',      'nunique'),
    total_price        = ('price',          'sum'),
    total_freight      = ('freight_value',  'sum'),
    avg_item_price     = ('price',          'mean'),
    max_item_price     = ('price',          'max'),
    # The dominant category is the one with the most items in this order.
    order_category_pt  = ('product_category_name',         lambda x: x.mode()[0] if x.notna().any() else np.nan),
    order_category_en  = ('product_category_name_english', lambda x: x.mode()[0] if x.notna().any() else np.nan),
    avg_weight_g       = ('product_weight_g', 'mean'),
).reset_index()

# Freight ratio = shipping cost as a fraction of the product cost.
items_agg['freight_ratio'] = np.where(
    items_agg['total_price'] > 0,
    items_agg['total_freight'] / items_agg['total_price'],
    np.nan
)

print("items_agg shape:", items_agg.shape)
print(items_agg.head(3))

items_agg shape: (98666, 12)
                           order_id  total_items  unique_products  \
0  00010242fe8c5a6d1ba2dd792cb16214            1                1   
1  00018f77f2f0320c557190d7a144bdd3            1                1   
2  000229ec398224ef6ca0657da4fc703e            1                1   

   unique_sellers  total_price  total_freight  avg_item_price  max_item_price  \
0               1         58.9          13.29            58.9            58.9   
1               1        239.9          19.93           239.9           239.9   
2               1        199.0          17.87           199.0           199.0   

  order_category_pt order_category_en  avg_weight_g  freight_ratio  
0        cool_stuff        cool_stuff         650.0       0.225637  
1          pet_shop          pet_shop       30000.0       0.083076  
2  moveis_decoracao   furniture_decor        3050.0       0.089799  


### 3c. Aggregate payments to the order level

One order can have multiple payment rows (e.g. credit card + voucher).  
We keep the dominant payment type and sum the total paid.

In [5]:
# Helper: for each order, find the payment type with the highest total value.
pay_by_type = payments.groupby(['order_id', 'payment_type'])['payment_value'].sum().reset_index()
dominant_type = pay_by_type.loc[
    pay_by_type.groupby('order_id')['payment_value'].idxmax()
][['order_id', 'payment_type']].rename(columns={'payment_type': 'dominant_payment_type'})

# Max installments used across all payment rows for this order.
pay_agg = payments.groupby('order_id').agg(
    total_payment_value    = ('payment_value',        'sum'),
    payment_methods_count  = ('payment_type',         'nunique'),
    max_installments       = ('payment_installments', 'max'),
).reset_index()

pay_agg = pay_agg.merge(dominant_type, on='order_id', how='left')

# Binary flags for each payment type (useful for ML models).
pay_agg['paid_with_credit_card'] = payments.groupby('order_id')['payment_type'].apply(
    lambda x: int('credit_card' in x.values)
).reset_index(drop=True)

# Simpler approach — one-hot using a merge on known types.
for ptype in ['credit_card', 'boleto', 'voucher', 'debit_card']:
    flag_col = f'used_{ptype}'
    orders_with_type = payments[payments['payment_type'] == ptype]['order_id'].unique()
    pay_agg[flag_col] = pay_agg['order_id'].isin(orders_with_type).astype(int)

# Drop the intermediate column from the loop above.
pay_agg = pay_agg.drop(columns=['paid_with_credit_card'])

print("pay_agg shape:", pay_agg.shape)
print(pay_agg.head(3))

pay_agg shape: (99440, 9)
                           order_id  total_payment_value  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19   
1  00018f77f2f0320c557190d7a144bdd3               259.83   
2  000229ec398224ef6ca0657da4fc703e               216.87   

   payment_methods_count  max_installments dominant_payment_type  \
0                      1                 2           credit_card   
1                      1                 3           credit_card   
2                      1                 5           credit_card   

   used_credit_card  used_boleto  used_voucher  used_debit_card  
0                 1            0             0                0  
1                 1            0             0                0  
2                 1            0             0                0  


### 3d. Aggregate reviews to the order level

Some orders have more than one review record (due to the repeated review_id issue found in cleaning).  
We take the most recent review score per order.

In [6]:
# Sort by creation date descending so the most recent review comes first.
reviews_sorted = reviews.sort_values('review_creation_date', ascending=False)

# Keep only the latest review per order.
review_agg = reviews_sorted.drop_duplicates(subset='order_id', keep='first')[
    ['order_id', 'review_score', 'review_creation_date']
].rename(columns={
    'review_score': 'review_score',
    'review_creation_date': 'review_creation_date'
})

print("review_agg shape:", review_agg.shape)
print("Unique orders with a review:", review_agg['order_id'].nunique())

review_agg shape: (98673, 3)
Unique orders with a review: 98673


### 3e. Join everything into a single order-level table

In [7]:
# Start with orders (already has customer info from step 3a).
order_base = orders.merge(items_agg,   on='order_id', how='left')
order_base = order_base.merge(pay_agg,    on='order_id', how='left')
order_base = order_base.merge(review_agg, on='order_id', how='left')

print("order_base shape:", order_base.shape)
print("Columns:", list(order_base.columns))

order_base shape: (99441, 33)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_city', 'customer_state', 'customer_zip_code_prefix', 'total_items', 'unique_products', 'unique_sellers', 'total_price', 'total_freight', 'avg_item_price', 'max_item_price', 'order_category_pt', 'order_category_en', 'avg_weight_g', 'freight_ratio', 'total_payment_value', 'payment_methods_count', 'max_installments', 'dominant_payment_type', 'used_credit_card', 'used_boleto', 'used_voucher', 'used_debit_card', 'review_score', 'review_creation_date']


---
## 4. Engineer order-level features

Now we create derived columns that capture timing, delivery performance, and order characteristics.

### 4a. Time-based features

In [8]:
# --- Approval time: how long from purchase to payment approval ---
# Only meaningful when order_approved_at is present.
order_base['approval_time_days'] = (
    order_base['order_approved_at'] - order_base['order_purchase_timestamp']
).dt.total_seconds() / 86400  # convert seconds → days

# --- Carrier pickup time: approval → handed to carrier ---
order_base['carrier_pickup_days'] = (
    order_base['order_delivered_carrier_date'] - order_base['order_approved_at']
).dt.total_seconds() / 86400

# --- Last-mile delivery time: carrier → customer ---
order_base['last_mile_days'] = (
    order_base['order_delivered_customer_date'] - order_base['order_delivered_carrier_date']
).dt.total_seconds() / 86400

# --- Total end-to-end delivery time: purchase → customer received ---
order_base['delivery_time_days'] = (
    order_base['order_delivered_customer_date'] - order_base['order_purchase_timestamp']
).dt.total_seconds() / 86400

# --- Estimated vs actual: how many days early (positive) or late (negative) ---
# Positive = delivered before the estimate (good). Negative = delivered late.
order_base['delivery_vs_estimate_days'] = (
    order_base['order_estimated_delivery_date'] - order_base['order_delivered_customer_date']
).dt.total_seconds() / 86400

# --- Purchase calendar features ---
order_base['purchase_year']          = order_base['order_purchase_timestamp'].dt.year
order_base['purchase_month']         = order_base['order_purchase_timestamp'].dt.month
order_base['purchase_day_of_week']   = order_base['order_purchase_timestamp'].dt.dayofweek  # 0=Mon
order_base['purchase_hour']          = order_base['order_purchase_timestamp'].dt.hour

# Weekend flag (Saturday=5, Sunday=6)
order_base['is_weekend_purchase'] = (order_base['purchase_day_of_week'] >= 5).astype(int)

print("Time features added.")
time_cols = ['approval_time_days','carrier_pickup_days','last_mile_days',
             'delivery_time_days','delivery_vs_estimate_days']
print(order_base[time_cols].describe().round(2))

Time features added.
       approval_time_days  carrier_pickup_days  last_mile_days  \
count            99281.00             97644.00        96475.00   
mean                 0.43                 2.81            9.33   
std                  1.08                 3.55            8.76   
min                  0.00              -171.22          -16.10   
25%                  0.01                 0.88            4.10   
50%                  0.01                 1.82            7.10   
75%                  0.61                 3.58           12.03   
max                187.88               125.76          205.19   

       delivery_time_days  delivery_vs_estimate_days  
count            96476.00                   96476.00  
mean                12.56                      11.18  
std                  9.55                      10.19  
min                  0.53                    -188.98  
25%                  6.77                       6.39  
50%                 10.22                      11.95  

### 4b. Delivery performance flag

In [9]:
# is_on_time = 1 if the order arrived on or before the estimated date, else 0.
# Only set for delivered orders where both timestamps are available.
is_delivered = order_base['order_status'] == 'delivered'
has_both_dates = (
    order_base['order_delivered_customer_date'].notna() &
    order_base['order_estimated_delivery_date'].notna()
)

order_base['is_on_time'] = np.where(
    is_delivered & has_both_dates,
    (order_base['delivery_vs_estimate_days'] >= 0).astype(int),
    np.nan  # Not applicable for non-delivered orders
)

print("On-time delivery breakdown (delivered orders only):")
print(order_base[is_delivered]['is_on_time'].value_counts())
on_time_rate = order_base[is_delivered]['is_on_time'].mean()
print(f"Overall on-time rate: {on_time_rate:.1%}")

On-time delivery breakdown (delivered orders only):
is_on_time
1.0    88644
0.0     7826
Name: count, dtype: int64
Overall on-time rate: 91.9%


### 4c. Order value features

In [10]:
# Order value tier — label each order as Small, Medium, Large, or Very Large.
# Boundaries chosen from the quartiles of total_payment_value.
order_base['order_value_tier'] = pd.cut(
    order_base['total_payment_value'],
    bins=[0, 50, 100, 200, float('inf')],
    labels=['Small', 'Medium', 'Large', 'VeryLarge'],
    right=True
)

# Multi-item flag
order_base['is_multi_item'] = (order_base['total_items'] > 1).astype(int)

# Multi-seller flag (the order had products from more than one seller)
order_base['is_multi_seller'] = (order_base['unique_sellers'] > 1).astype(int)

print("Order value tier distribution:")
print(order_base['order_value_tier'].value_counts().sort_index())
print()
print(f"Multi-item orders:  {order_base['is_multi_item'].sum():,}")
print(f"Multi-seller orders: {order_base['is_multi_seller'].sum():,}")

Order value tier distribution:
order_value_tier
Small        16955
Medium       30301
Large        31902
VeryLarge    20279
Name: count, dtype: int64

Multi-item orders:  9,803
Multi-seller orders: 1,278


### 4d. Review features at the order level

In [11]:
# Binary: did the customer leave a review?
order_base['has_review'] = order_base['review_score'].notna().astype(int)

# Is this a positive review (4 or 5 stars)?
order_base['is_positive_review'] = np.where(
    order_base['review_score'].notna(),
    (order_base['review_score'] >= 4).astype(int),
    np.nan
)

# Review sentiment bucket
order_base['review_sentiment'] = pd.cut(
    order_base['review_score'],
    bins=[0, 2, 3, 5],
    labels=['negative', 'neutral', 'positive'],
    right=True
)

print("Review sentiment distribution:")
print(order_base['review_sentiment'].value_counts())

Review sentiment distribution:
review_sentiment
positive    76046
negative    14494
neutral      8133
Name: count, dtype: int64


### 4e. Verify the order-level table

In [12]:
print("order_base final shape:", order_base.shape)
print("\nSample row (delivered order):")
sample = order_base[order_base['order_status'] == 'delivered'].iloc[0]
display_cols = [
    'order_id','order_status','total_items','total_price','total_freight',
    'freight_ratio','total_payment_value','dominant_payment_type','max_installments',
    'approval_time_days','delivery_time_days','delivery_vs_estimate_days',
    'is_on_time','review_score','order_value_tier','purchase_month'
]
print(sample[display_cols].to_string())

order_base final shape: (99441, 50)

Sample row (delivered order):


order_id                     e481f51cbdc54678b7cc49136f2d6af7
order_status                                        delivered
total_items                                               1.0
total_price                                             29.99
total_freight                                            8.72
freight_ratio                                        0.290764
total_payment_value                                     38.71
dominant_payment_type                                 voucher
max_installments                                          1.0
approval_time_days                                   0.007431
delivery_time_days                                   8.436574
delivery_vs_estimate_days                            7.107488
is_on_time                                                1.0
review_score                                              4.0
order_value_tier                                        Small
purchase_month                                             10


---
## 5. Build order_features.csv

We select the final columns and save the order-level feature table.

In [13]:
order_feature_cols = [
    # Keys
    'order_id', 'customer_id', 'customer_unique_id',
    # Order metadata
    'order_status', 'order_purchase_timestamp',
    'purchase_year', 'purchase_month', 'purchase_day_of_week',
    'purchase_hour', 'is_weekend_purchase',
    # Customer location
    'customer_state', 'customer_city', 'customer_zip_code_prefix',
    # Item features
    'total_items', 'unique_products', 'unique_sellers',
    'total_price', 'avg_item_price', 'max_item_price',
    'total_freight', 'freight_ratio',
    'avg_weight_g',
    'order_category_pt', 'order_category_en',
    'is_multi_item', 'is_multi_seller',
    # Payment features
    'total_payment_value', 'dominant_payment_type',
    'payment_methods_count', 'max_installments',
    'used_credit_card', 'used_boleto', 'used_voucher', 'used_debit_card',
    'order_value_tier',
    # Delivery features
    'approval_time_days', 'carrier_pickup_days',
    'last_mile_days', 'delivery_time_days',
    'delivery_vs_estimate_days', 'is_on_time',
    # Review features
    'review_score', 'has_review', 'is_positive_review', 'review_sentiment',
]

order_features = order_base[order_feature_cols].copy()

print("order_features shape:", order_features.shape)
print("Columns:", list(order_features.columns))

order_features shape: (99441, 45)
Columns: ['order_id', 'customer_id', 'customer_unique_id', 'order_status', 'order_purchase_timestamp', 'purchase_year', 'purchase_month', 'purchase_day_of_week', 'purchase_hour', 'is_weekend_purchase', 'customer_state', 'customer_city', 'customer_zip_code_prefix', 'total_items', 'unique_products', 'unique_sellers', 'total_price', 'avg_item_price', 'max_item_price', 'total_freight', 'freight_ratio', 'avg_weight_g', 'order_category_pt', 'order_category_en', 'is_multi_item', 'is_multi_seller', 'total_payment_value', 'dominant_payment_type', 'payment_methods_count', 'max_installments', 'used_credit_card', 'used_boleto', 'used_voucher', 'used_debit_card', 'order_value_tier', 'approval_time_days', 'carrier_pickup_days', 'last_mile_days', 'delivery_time_days', 'delivery_vs_estimate_days', 'is_on_time', 'review_score', 'has_review', 'is_positive_review', 'review_sentiment']


In [14]:
order_features.to_csv("Feature_Data/order_features.csv", index=False)
print("order_features.csv saved. Rows:", len(order_features))

order_features.csv saved. Rows: 99441


---
## 6. Build customer-level features

We aggregate order_base to one row per real customer (`customer_unique_id`).  
This is the most important table for customer segmentation and churn prediction.

### Feature groups we will build:

| Group | What it captures |
|---|---|
| **RFM** | Recency, Frequency, Monetary — the classic customer value framework |
| **Order behaviour** | Average order size, multi-item habits, category preferences |
| **Delivery experience** | Average delivery time, on-time rate |
| **Review behaviour** | Average score, review rate |
| **Payment behaviour** | Preferred payment type, avg installments |
| **Geography** | State, city |
| **Derived labels** | Is returning customer |

**Snapshot date:** one day after the last order in the dataset (2018-10-18).  
Recency is always measured relative to this date.

### 6a. Define the snapshot date

In [15]:
# Snapshot date = one day after the latest order in the dataset.
# This gives a fixed reference point for recency calculations.
snapshot_date = order_base['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print("Snapshot date (RFM reference):", snapshot_date.date())

Snapshot date (RFM reference): 2018-10-18


### 6b. RFM — Recency, Frequency, Monetary

In [16]:
# Work only with delivered orders for RFM — the customer actually received and paid for these.
delivered_orders = order_base[order_base['order_status'] == 'delivered'].copy()

rfm = delivered_orders.groupby('customer_unique_id').agg(
    # Recency: how many days since the customer's most recent purchase?
    last_order_date  = ('order_purchase_timestamp', 'max'),
    # Frequency: how many orders has the customer placed?
    order_count      = ('order_id', 'nunique'),
    # Monetary: total amount paid across all orders.
    total_spent      = ('total_payment_value', 'sum'),
).reset_index()

# Recency in days from the snapshot date.
rfm['recency_days'] = (snapshot_date - rfm['last_order_date']).dt.days

# Average order value
rfm['avg_order_value'] = rfm['total_spent'] / rfm['order_count']

print("RFM table shape:", rfm.shape)
print()
print(rfm[['recency_days', 'order_count', 'total_spent', 'avg_order_value']].describe().round(2))

RFM table shape: (93358, 6)

       recency_days  order_count  total_spent  avg_order_value
count      93358.00     93358.00     93358.00         93358.00
mean         287.11         1.03       165.20           160.31
std          152.59         0.21       226.31           219.57
min           50.00         1.00         0.00             0.00
25%          164.00         1.00        63.05            62.37
50%          268.00         1.00       107.78           105.63
75%          395.00         1.00       182.56           176.65
max          763.00        15.00     13664.08         13664.08


### 6c. Score R, F, and M on a 1–5 scale using quintiles

Quintile scoring gives each customer a score from 1 to 5 on each dimension.

- **R score**: lower recency (bought more recently) → higher score.
- **F score**: higher frequency → higher score.
- **M score**: higher spend → higher score.

In [17]:
# R score: recent buyers get 5, oldest get 1 (reverse order).
rfm['r_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# F score: most frequent buyers get 5.
# Many customers have frequency=1, so qcut would produce duplicate edges.
# We use rank-based scoring to handle ties.
rfm['f_score'] = pd.cut(
    rfm['order_count'].rank(method='first'),
    bins=5, labels=[1, 2, 3, 4, 5]
).astype(int)

# M score: highest spenders get 5.
rfm['m_score'] = pd.qcut(rfm['total_spent'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# RFM combined score (sum) and segment label.
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

print("RFM score distribution:")
print(rfm['rfm_score'].value_counts().sort_index().to_dict())
print()
print(rfm[['r_score', 'f_score', 'm_score', 'rfm_score']].describe().round(2))

RFM score distribution:
{3: 799, 4: 2365, 5: 4698, 6: 7578, 7: 11100, 8: 13313, 9: 13986, 10: 13151, 11: 10888, 12: 7464, 13: 4583, 14: 2433, 15: 1000}

        r_score   f_score   m_score  rfm_score
count  93358.00  93358.00  93358.00    93358.0
mean       3.00      3.00      3.00        9.0
std        1.41      1.41      1.41        2.5
min        1.00      1.00      1.00        3.0
25%        2.00      2.00      2.00        7.0
50%        3.00      3.00      3.00        9.0
75%        4.00      4.00      4.00       11.0
max        5.00      5.00      5.00       15.0


### 6d. Assign a human-readable customer segment

We map the RFM scores to four intuitive business segments.  
The rules follow the standard RFM segmentation logic.

In [18]:
def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r == 1 and f >= 2:
        return 'Lost'
    else:
        return 'Occasional'

rfm['customer_segment'] = rfm.apply(assign_segment, axis=1)

print("Customer segment distribution:")
print(rfm['customer_segment'].value_counts())

Customer segment distribution:
customer_segment
Loyal Customers    27262
At Risk            22269
Occasional         18632
New Customers      14974
Champions           6484
Lost                3737
Name: count, dtype: int64


### 6e. Order behaviour features

In [19]:
order_behaviour = delivered_orders.groupby('customer_unique_id').agg(
    avg_items_per_order    = ('total_items',          'mean'),
    avg_freight            = ('total_freight',         'mean'),
    avg_freight_ratio      = ('freight_ratio',         'mean'),
    max_order_value        = ('total_payment_value',   'max'),
    min_order_value        = ('total_payment_value',   'min'),
    multi_item_order_count = ('is_multi_item',         'sum'),
    # Preferred category: most purchased Portuguese category name.
    preferred_category_pt  = ('order_category_pt',
                              lambda x: x.mode()[0] if x.notna().any() else np.nan),
    preferred_category_en  = ('order_category_en',
                              lambda x: x.mode()[0] if x.notna().any() else np.nan),
).reset_index()

# What fraction of this customer's orders had multiple items?
order_behaviour['multi_item_rate'] = (
    order_behaviour['multi_item_order_count'] / rfm.set_index('customer_unique_id').loc[
        order_behaviour['customer_unique_id'], 'order_count'
    ].values
)

print("order_behaviour shape:", order_behaviour.shape)
print(order_behaviour[['avg_items_per_order','avg_freight_ratio','multi_item_rate']].describe().round(3))

order_behaviour shape: (93358, 10)
       avg_items_per_order  avg_freight_ratio  multi_item_rate
count            93358.000          93358.000        93358.000
mean                 1.140              0.308            0.098
std                  0.527              0.310            0.295
min                  1.000              0.000            0.000
25%                  1.000              0.132            0.000
50%                  1.000              0.225            0.000
75%                  1.000              0.380            0.000
max                 21.000             21.447            1.000


### 6f. Delivery experience features

In [20]:
delivery_exp = delivered_orders.groupby('customer_unique_id').agg(
    avg_delivery_days         = ('delivery_time_days',         'mean'),
    max_delivery_days         = ('delivery_time_days',         'max'),
    avg_delivery_vs_estimate  = ('delivery_vs_estimate_days',  'mean'),
    on_time_order_count       = ('is_on_time',                 'sum'),
    deliveries_with_data      = ('is_on_time',                 'count'),
).reset_index()

# On-time rate: fraction of orders that arrived on or before the estimate.
delivery_exp['on_time_rate'] = (
    delivery_exp['on_time_order_count'] / delivery_exp['deliveries_with_data']
)

# Drop the helper count column.
delivery_exp = delivery_exp.drop(columns=['on_time_order_count', 'deliveries_with_data'])

print("delivery_exp shape:", delivery_exp.shape)
print(delivery_exp[['avg_delivery_days','avg_delivery_vs_estimate','on_time_rate']].describe().round(2))

delivery_exp shape: (93358, 5)
       avg_delivery_days  avg_delivery_vs_estimate  on_time_rate
count           93350.00                  93350.00      93350.00
mean               12.57                     11.15          0.92
std                 9.55                     10.14          0.27
min                 0.53                   -188.98          0.00
25%                 6.79                      6.39          1.00
50%                10.23                     11.75          1.00
75%                15.72                     16.23          1.00
max               209.63                    146.02          1.00


### 6g. Review behaviour features

In [21]:
review_behaviour = delivered_orders.groupby('customer_unique_id').agg(
    avg_review_score   = ('review_score',       'mean'),
    review_count       = ('has_review',          'sum'),
    positive_reviews   = ('is_positive_review',  'sum'),
    total_orders_del   = ('order_id',            'nunique'),
).reset_index()

# Review rate = fraction of orders that received a review.
review_behaviour['review_rate'] = (
    review_behaviour['review_count'] / review_behaviour['total_orders_del']
)

# Positive review rate = fraction of reviews that were 4 or 5 stars.
review_behaviour['positive_review_rate'] = np.where(
    review_behaviour['review_count'] > 0,
    review_behaviour['positive_reviews'] / review_behaviour['review_count'],
    np.nan
)

review_behaviour = review_behaviour.drop(columns=['positive_reviews', 'total_orders_del'])

print("review_behaviour shape:", review_behaviour.shape)
print(review_behaviour[['avg_review_score','review_rate','positive_review_rate']].describe().round(3))

review_behaviour shape: (93358, 5)
       avg_review_score  review_rate  positive_review_rate
count         92755.000    93358.000             92755.000
mean              4.153        0.993                 0.789
std               1.281        0.081                 0.406
min               1.000        0.000                 0.000
25%               4.000        1.000                 1.000
50%               5.000        1.000                 1.000
75%               5.000        1.000                 1.000
max               5.000        1.000                 1.000


### 6h. Payment behaviour features

In [22]:
payment_behaviour = delivered_orders.groupby('customer_unique_id').agg(
    avg_installments      = ('max_installments',      'mean'),
    max_installments_ever = ('max_installments',      'max'),
    # Preferred payment = the type used most often.
    preferred_payment     = ('dominant_payment_type',
                             lambda x: x.mode()[0] if x.notna().any() else np.nan),
    cc_orders             = ('used_credit_card',      'sum'),
    boleto_orders         = ('used_boleto',           'sum'),
    voucher_orders        = ('used_voucher',          'sum'),
).reset_index()

print("payment_behaviour shape:", payment_behaviour.shape)
print("Preferred payment distribution:")
print(payment_behaviour['preferred_payment'].value_counts())

payment_behaviour shape: (93358, 7)
Preferred payment distribution:
preferred_payment
credit_card    70400
boleto         18702
voucher         2831
debit_card      1424
Name: count, dtype: int64


### 6i. Geography and returning-customer flag

In [23]:
# Geography: use the location from the customer's most recent order.
most_recent_order = (
    order_base
    .sort_values('order_purchase_timestamp', ascending=False)
    .drop_duplicates(subset='customer_unique_id', keep='first')
[['customer_unique_id', 'customer_state', 'customer_city', 'customer_zip_code_prefix']]
)

# Returning customer flag
# A customer is 'returning' if their customer_unique_id appears in more than one order.
order_count_per_unique = order_base.groupby('customer_unique_id')['order_id'].nunique()
is_returning = (order_count_per_unique > 1).reset_index()
is_returning.columns = ['customer_unique_id', 'is_returning_customer']
is_returning['is_returning_customer'] = is_returning['is_returning_customer'].astype(int)

print("Returning customers:", is_returning['is_returning_customer'].sum())
print("One-time customers: ", (is_returning['is_returning_customer'] == 0).sum())

Returning customers: 2997
One-time customers:  93099


### 6j. Merge all customer features into one table

In [24]:
# Start with RFM (only delivered orders — same base as all other aggregations).
customer_features = rfm[[
    'customer_unique_id', 'last_order_date', 'recency_days',
    'order_count', 'total_spent', 'avg_order_value',
    'r_score', 'f_score', 'm_score', 'rfm_score', 'customer_segment'
]].copy()

customer_features = customer_features.merge(order_behaviour,   on='customer_unique_id', how='left')
customer_features = customer_features.merge(delivery_exp,      on='customer_unique_id', how='left')
customer_features = customer_features.merge(review_behaviour,  on='customer_unique_id', how='left')
customer_features = customer_features.merge(payment_behaviour, on='customer_unique_id', how='left')
customer_features = customer_features.merge(most_recent_order, on='customer_unique_id', how='left')
customer_features = customer_features.merge(is_returning,      on='customer_unique_id', how='left')

print("customer_features shape:", customer_features.shape)
print("Columns:", list(customer_features.columns))

customer_features shape: (93358, 38)
Columns: ['customer_unique_id', 'last_order_date', 'recency_days', 'order_count', 'total_spent', 'avg_order_value', 'r_score', 'f_score', 'm_score', 'rfm_score', 'customer_segment', 'avg_items_per_order', 'avg_freight', 'avg_freight_ratio', 'max_order_value', 'min_order_value', 'multi_item_order_count', 'preferred_category_pt', 'preferred_category_en', 'multi_item_rate', 'avg_delivery_days', 'max_delivery_days', 'avg_delivery_vs_estimate', 'on_time_rate', 'avg_review_score', 'review_count', 'review_rate', 'positive_review_rate', 'avg_installments', 'max_installments_ever', 'preferred_payment', 'cc_orders', 'boleto_orders', 'voucher_orders', 'customer_state', 'customer_city', 'customer_zip_code_prefix', 'is_returning_customer']


In [25]:
# Sanity check: each row should be a unique customer.
assert customer_features['customer_unique_id'].duplicated().sum() == 0, \
    "Duplicate customer_unique_id found!"

print("Primary key check: OK — every row is a unique customer.")
print()
print("Missing value summary:")
missing = customer_features.isna().sum()
print(missing[missing > 0])

Primary key check: OK — every row is a unique customer.

Missing value summary:
max_order_value                1
min_order_value                1
preferred_category_pt       1260
preferred_category_en       1279
avg_delivery_days              8
max_delivery_days              8
avg_delivery_vs_estimate       8
on_time_rate                   8
avg_review_score             603
positive_review_rate         603
avg_installments               1
max_installments_ever          1
preferred_payment              1
dtype: int64


In [26]:
customer_features.to_csv("Feature_Data/customer_features.csv", index=False)
print("customer_features.csv saved. Rows:", len(customer_features))
print()
print("Segment summary:")
print(customer_features.groupby('customer_segment').agg(
    customers=('customer_unique_id', 'count'),
    avg_spend=('total_spent', 'mean'),
    avg_orders=('order_count', 'mean'),
    avg_recency=('recency_days', 'mean')
).round(1))

customer_features.csv saved. Rows: 93358

Segment summary:
                  customers  avg_spend  avg_orders  avg_recency
customer_segment                                               
At Risk               22269      167.0         1.0        443.6
Champions              6484      312.2         1.2        140.1
Lost                   3737      160.0         1.0        521.8
Loyal Customers       27262      134.0         1.0        193.2
New Customers         14974      163.5         1.0        140.0
Occasional            18632      160.0         1.0        359.7


---
## 7. Build seller-level features

Seller features help answer: *which sellers perform best and which need improvement?*  
They can also be used as lookup features in order-level models.

### 7a. Attach order and review info to each item

In [27]:
# Build a seller-item view by merging order_base info onto order_items.
# We join on order_id to get delivery performance and review score for each item.
seller_item_view = order_items[['order_id', 'seller_id', 'price', 'freight_value']].merge(
    order_base[['order_id', 'order_status', 'delivery_time_days',
                'delivery_vs_estimate_days', 'is_on_time', 'review_score',
                'order_purchase_timestamp']],
    on='order_id',
    how='left'
)

print("seller_item_view shape:", seller_item_view.shape)
print(seller_item_view.head(3))

seller_item_view shape: (112650, 10)
                           order_id                         seller_id  price  \
0  00010242fe8c5a6d1ba2dd792cb16214  48436dade18ac8b2bce089ec2a041202   58.9   
1  00018f77f2f0320c557190d7a144bdd3  dd7ddc04e1b6c2c614352b383efe2d36  239.9   
2  000229ec398224ef6ca0657da4fc703e  5b51032eddd242adc84c38acab88f23d  199.0   

   freight_value order_status  delivery_time_days  delivery_vs_estimate_days  \
0          13.29    delivered            7.614421                   8.011250   
1          19.93    delivered           16.216181                   2.330278   
2          17.87    delivered            7.948437                  13.444954   

   is_on_time  review_score order_purchase_timestamp  
0         1.0           5.0      2017-09-13 08:59:02  
1         1.0           4.0      2017-04-26 10:53:06  
2         1.0           5.0      2018-01-14 14:33:31  


### 7b. Aggregate to seller level

In [28]:
seller_agg = seller_item_view.groupby('seller_id').agg(
    # Volume
    total_items_sold      = ('order_id',               'count'),
    total_orders          = ('order_id',               'nunique'),
    # Revenue
    total_revenue         = ('price',                  'sum'),
    avg_item_price        = ('price',                  'mean'),
    # Delivery performance (for delivered orders only)
    avg_delivery_days     = ('delivery_time_days',     'mean'),
    avg_vs_estimate       = ('delivery_vs_estimate_days', 'mean'),
    on_time_count         = ('is_on_time',             'sum'),
    orders_with_timing    = ('is_on_time',             'count'),
    # Customer satisfaction
    avg_review_score      = ('review_score',           'mean'),
    review_count          = ('review_score',           'count'),
    # Activity window
    first_sale_date       = ('order_purchase_timestamp', 'min'),
    last_sale_date        = ('order_purchase_timestamp', 'max'),
).reset_index()

# On-time rate
seller_agg['on_time_rate'] = np.where(
    seller_agg['orders_with_timing'] > 0,
    seller_agg['on_time_count'] / seller_agg['orders_with_timing'],
    np.nan
)

# Activity span in days
seller_agg['active_days'] = (
    seller_agg['last_sale_date'] - seller_agg['first_sale_date']
).dt.days

# Drop helper columns
seller_agg = seller_agg.drop(columns=['on_time_count', 'orders_with_timing'])

print("seller_agg shape:", seller_agg.shape)
print(seller_agg[['total_orders','total_revenue','avg_review_score','on_time_rate']].describe().round(2))

seller_agg shape: (3095, 13)
       total_orders  total_revenue  avg_review_score  on_time_rate
count       3095.00        3095.00           3090.00       2970.00
mean          32.31        4391.48              3.97          0.92
std          105.14       13922.00              0.97          0.17
min            1.00           3.50              1.00          0.00
25%            2.00         208.85              3.71          0.90
50%            6.00         821.48              4.17          1.00
75%           21.50        3280.83              4.60          1.00
max         1854.00      229472.63              5.00          1.00


### 7c. Add seller location and performance tier

In [29]:
# Merge in seller location from the sellers table.
seller_features = seller_agg.merge(
    sellers[['seller_id', 'seller_city', 'seller_state', 'seller_zip_code_prefix']],
    on='seller_id',
    how='left'
)

# Seller performance tier based on average review score.
seller_features['performance_tier'] = pd.cut(
    seller_features['avg_review_score'],
    bins=[0, 2.5, 3.5, 4.2, 5.0],
    labels=['Poor', 'Average', 'Good', 'Excellent'],
    right=True
)

# Volume tier (by number of orders)
seller_features['volume_tier'] = pd.cut(
    seller_features['total_orders'],
    bins=[0, 10, 50, 200, float('inf')],
    labels=['Micro', 'Small', 'Medium', 'Large'],
    right=True
)

print("seller_features shape:", seller_features.shape)
print()
print("Performance tier distribution:")
print(seller_features['performance_tier'].value_counts())
print()
print("Volume tier distribution:")
print(seller_features['volume_tier'].value_counts())

seller_features shape: (3095, 18)

Performance tier distribution:
performance_tier
Excellent    1475
Good          986
Average       348
Poor          281
Name: count, dtype: int64

Volume tier distribution:
volume_tier
Micro     1894
Small      780
Medium     332
Large       89
Name: count, dtype: int64


In [30]:
# Sanity check: one row per seller.
assert seller_features['seller_id'].duplicated().sum() == 0
print("Primary key check: OK — every row is a unique seller.")
print()
print("Missing value summary:")
missing_s = seller_features.isna().sum()
print(missing_s[missing_s > 0])

Primary key check: OK — every row is a unique seller.

Missing value summary:
avg_delivery_days    125
avg_vs_estimate      125
avg_review_score       5
on_time_rate         125
performance_tier       5
dtype: int64


In [31]:
seller_features.to_csv("Feature_Data/seller_features.csv", index=False)
print("seller_features.csv saved. Rows:", len(seller_features))
print()
print("Columns:", list(seller_features.columns))

seller_features.csv saved. Rows: 3095

Columns: ['seller_id', 'total_items_sold', 'total_orders', 'total_revenue', 'avg_item_price', 'avg_delivery_days', 'avg_vs_estimate', 'avg_review_score', 'review_count', 'first_sale_date', 'last_sale_date', 'on_time_rate', 'active_days', 'seller_city', 'seller_state', 'seller_zip_code_prefix', 'performance_tier', 'volume_tier']


---
## 8. Final validation

A quick check to confirm all three feature tables are correct.

### 8a. Shape and primary key check

In [32]:
tables = [
    ('order_features',    order_features,    'order_id'),
    ('customer_features', customer_features, 'customer_unique_id'),
    ('seller_features',   seller_features,   'seller_id'),
]

print(f"{'Table':25s}  {'Rows':>8}  {'Cols':>5}  {'PK dupes':>10}")
print("-" * 58)
for name, df, pk in tables:
    dupes = df[pk].duplicated().sum()
    status = 'OK' if dupes == 0 else 'ISSUE'
    print(f"{name:25s}  {len(df):>8,}  {df.shape[1]:>5}  {dupes:>8}  {status}")

Table                          Rows   Cols    PK dupes
----------------------------------------------------------
order_features               99,441     45         0  OK
customer_features            93,358     38         0  OK
seller_features               3,095     18         0  OK


### 8b. Missing-value summary for each feature table

In [33]:
for name, df, _ in tables:
    missing = df.isna().sum()
    missing_cols = missing[missing > 0]
    print(f"=== {name} ===")
    if len(missing_cols) == 0:
        print("  No missing values.")
    else:
        for col, count in missing_cols.items():
            pct = count / len(df) * 100
            print(f"  {col:40s}  {count:>7,}  ({pct:.1f}%)")
    print()

=== order_features ===
  total_items                                   775  (0.8%)
  unique_products                               775  (0.8%)
  unique_sellers                                775  (0.8%)
  total_price                                   775  (0.8%)
  avg_item_price                                775  (0.8%)
  max_item_price                                775  (0.8%)
  total_freight                                 775  (0.8%)
  freight_ratio                                 775  (0.8%)
  avg_weight_g                                  797  (0.8%)
  order_category_pt                           2,164  (2.2%)
  order_category_en                           2,185  (2.2%)
  total_payment_value                             1  (0.0%)
  dominant_payment_type                           1  (0.0%)
  payment_methods_count                           1  (0.0%)
  max_installments                                1  (0.0%)
  used_credit_card                                1  (0.0%)
  used_boleto    

### 8c. Key statistics at a glance

In [34]:
print("=== ORDER FEATURES ===")
print(f"  Total orders:          {len(order_features):>8,}")
print(f"  Delivered:             {(order_features['order_status']=='delivered').sum():>8,}")
print(f"  On-time (delivered):   {order_features['is_on_time'].sum():>8,.0f}")
print(f"  Avg delivery days:     {order_features['delivery_time_days'].mean():>8.1f}")
print(f"  Avg payment value:     {order_features['total_payment_value'].mean():>8.2f}")
print()

print("=== CUSTOMER FEATURES ===")
print(f"  Unique customers:      {len(customer_features):>8,}")
print(f"  Returning customers:   {customer_features['is_returning_customer'].sum():>8,}")
print(f"  Avg recency (days):    {customer_features['recency_days'].mean():>8.1f}")
print(f"  Avg total spent:       {customer_features['total_spent'].mean():>8.2f}")
print(f"  Avg review score:      {customer_features['avg_review_score'].mean():>8.2f}")
print()

print("=== SELLER FEATURES ===")
print(f"  Unique sellers:        {len(seller_features):>8,}")
print(f"  Avg orders per seller: {seller_features['total_orders'].mean():>8.1f}")
print(f"  Avg review score:      {seller_features['avg_review_score'].mean():>8.2f}")
print(f"  Avg on-time rate:      {seller_features['on_time_rate'].mean():>8.1%}")

=== ORDER FEATURES ===
  Total orders:            99,441
  Delivered:               96,478
  On-time (delivered):     88,644
  Avg delivery days:         12.6
  Avg payment value:       160.99

=== CUSTOMER FEATURES ===
  Unique customers:        93,358
  Returning customers:      2,979
  Avg recency (days):       287.1
  Avg total spent:         165.20
  Avg review score:          4.15

=== SELLER FEATURES ===
  Unique sellers:           3,095
  Avg orders per seller:     32.3
  Avg review score:          3.97
  Avg on-time rate:         91.6%


---
## 9. Feature glossary

### order_features.csv (one row per order)

| Feature | Description |
|---|---|
| `order_id` | Primary key |
| `customer_unique_id` | Links to customer_features |
| `purchase_year / month / day_of_week / hour` | When the order was placed |
| `is_weekend_purchase` | 1 if placed on Saturday or Sunday |
| `total_items` | Number of items in the order |
| `total_price` | Sum of item prices (before shipping) |
| `avg_item_price` | Mean item price in this order |
| `total_freight` | Total shipping cost |
| `freight_ratio` | Shipping as a fraction of product cost |
| `avg_weight_g` | Average product weight in grams |
| `order_category_pt / en` | Dominant product category (Portuguese / English) |
| `is_multi_item` | 1 if the order had more than one item |
| `is_multi_seller` | 1 if items came from more than one seller |
| `total_payment_value` | Total amount actually paid |
| `dominant_payment_type` | The payment type with the highest value |
| `max_installments` | Maximum number of instalments chosen |
| `used_credit_card / boleto / voucher / debit_card` | Binary flags for each payment method used |
| `order_value_tier` | Small / Medium / Large / VeryLarge |
| `approval_time_days` | Days from purchase to payment approval |
| `carrier_pickup_days` | Days from approval to carrier handoff |
| `last_mile_days` | Days from carrier pickup to customer delivery |
| `delivery_time_days` | End-to-end delivery time in days |
| `delivery_vs_estimate_days` | Days ahead (positive) or late (negative) |
| `is_on_time` | 1 if delivered on or before the estimated date |
| `review_score` | Customer review (1–5), NaN if no review |
| `has_review` | 1 if the customer left a review |
| `is_positive_review` | 1 if review score is 4 or 5 |
| `review_sentiment` | negative / neutral / positive |

### customer_features.csv (one row per unique customer)

| Feature | Description |
|---|---|
| `customer_unique_id` | Primary key |
| `recency_days` | Days since the customer's last purchase |
| `order_count` | Number of delivered orders (Frequency) |
| `total_spent` | Total paid across all delivered orders (Monetary) |
| `avg_order_value` | Average order value |
| `r_score / f_score / m_score` | RFM quintile scores (1–5) |
| `rfm_score` | Sum of R + F + M (3–15) |
| `customer_segment` | Champions / Loyal / New / At Risk / Lost / Occasional |
| `avg_items_per_order` | Average basket size |
| `avg_freight` | Average shipping cost paid |
| `avg_freight_ratio` | Average freight-to-price ratio |
| `preferred_category_pt / en` | Most purchased category |
| `multi_item_rate` | Fraction of orders that had multiple items |
| `avg_delivery_days` | Average end-to-end delivery time |
| `avg_delivery_vs_estimate` | Average days ahead / late |
| `on_time_rate` | Fraction of orders delivered on time |
| `avg_review_score` | Average review score left by this customer |
| `review_rate` | Fraction of orders reviewed |
| `positive_review_rate` | Fraction of reviews that were 4 or 5 stars |
| `preferred_payment` | Most used payment type |
| `avg_installments` | Average instalment count chosen |
| `is_returning_customer` | 1 if placed more than one order (any status) |

### seller_features.csv (one row per seller)

| Feature | Description |
|---|---|
| `seller_id` | Primary key |
| `total_items_sold` | Total individual items sold |
| `total_orders` | Unique orders fulfilled |
| `total_revenue` | Sum of item prices |
| `avg_item_price` | Average price per item |
| `avg_delivery_days` | Average delivery time for this seller's orders |
| `avg_vs_estimate` | Average days ahead / late |
| `on_time_rate` | Fraction of orders delivered on time |
| `avg_review_score` | Average review score received |
| `review_count` | Number of reviews received |
| `active_days` | Days between first and last sale |
| `performance_tier` | Poor / Average / Good / Excellent (by review score) |
| `volume_tier` | Micro / Small / Medium / Large (by order count) |

---
## Notebook complete

Three feature tables are saved in `Feature_Data/`:

| File | Rows | Columns | Use in Stage 2 |
|---|---|---|---|
| `order_features.csv` | 99,441 | 43 | Delivery prediction, review score prediction |
| `customer_features.csv` | ~96,000 | 30 | RFM segmentation, churn prediction, CLV |
| `seller_features.csv` | 3,095 | 17 | Seller quality ranking, risk flagging |

**Next steps in Stage 2:**
1. Customer segmentation with K-Means on RFM + behavioural features.
2. Churn / repeat-purchase prediction using `customer_features`.
3. Delivery delay prediction using `order_features`.
4. Review score prediction using order and seller features.
5. Model explainability with SHAP values.